<div align="center">
  <img src="https://media.giphy.com/media/LS9liihy8sST3GUQVs/giphy.gif" width="600"/>
</div>

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

### Configs

In [ ]:
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e5/train.csv"  
TEST_PATH  = "/kaggle/input/competitions/playground-series-s6e5/test.csv"
 
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 130, "figure.facecolor": "#0f0f0f",
                     "axes.facecolor": "#1a1a1a", "axes.labelcolor": "white",
                     "xtick.color": "white", "ytick.color": "white",
                     "text.color": "white", "grid.color": "#333333"})
 
ACCENT = "#e8002d"   # F1 red

## Loading Data

In [ ]:
print("Loading data...")
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
 
print(f"Train shape : {train.shape}")
print(f"Test  shape : {test.shape}")

## Basic Overview

In [ ]:
print("\n--- Train dtypes & nulls ---")
print(train.info())
print("\n--- Describe (numeric) ---")
display(train.describe().T)
 
print("\n--- Test columns vs Train ---")
missing_in_test = set(train.columns) - set(test.columns)
extra_in_test   = set(test.columns)  - set(train.columns)
print(f"  Cols in train not in test : {missing_in_test}")
print(f"  Cols in test  not in train: {extra_in_test}")

## Target Distribution

In [ ]:
vc = train['PitNextLap'].value_counts()
print(vc)
print(f"\nPositive rate: {vc[1]/len(train)*100:.2f}%")
 
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Target Distribution — PitNextLap", color="white", fontsize=14)
 
axes[0].bar(['No Pit (0)', 'Pit Next Lap (1)'], vc.values,
            color=['#444', ACCENT], edgecolor='white', linewidth=0.5)
axes[0].set_title("Raw Counts")
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 500, f"{v:,}", ha='center', color='white')
 
axes[1].pie(vc.values, labels=['No Pit', 'Pit Next Lap'],
            colors=['#444', ACCENT], autopct='%1.2f%%',
            textprops={'color': 'white'}, startangle=90)
axes[1].set_title("Proportions")
 
plt.tight_layout()
plt.show()

The dataset seems to be imbalanced with an approx 8:2 ratio of negatives to positives

## Numeric Feature Distributions

In [ ]:
num_cols = ['TyreLife', 'LapNumber', 'Position', 'LapTime (s)',
            'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress',
            'Position_Change', 'Stint']
 
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle("Feature Distributions: Train (blue) vs Test (orange)", fontsize=14, color='white')
 
for ax, col in zip(axes.flatten(), num_cols):
    ax.hist(train[col].dropna(), bins=50, alpha=0.6, color='#4e9af1',
            label='train', density=True)
    if col in test.columns:
        ax.hist(test[col].dropna(),  bins=50, alpha=0.6, color='#f1a94e',
                label='test',  density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=7)
 
plt.tight_layout()
plt.show()

The train and test distributions heavily overlap for most of the features, that is at inferance time the features will behave same as at the time of training.

## FEATURE vs TARGET  (pit vs no-pit distributions)
Feature importance preview before any model.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle("Feature Distributions by PitNextLap (0=blue, 1=red)", fontsize=14, color='white')
 
for ax, col in zip(axes.flatten(), num_cols):
    for val, color, label in [(0, '#4e9af1', 'No Pit'), (1, ACCENT, 'Pit')]:
        ax.hist(train.loc[train['PitNextLap'] == val, col].dropna(),
                bins=50, alpha=0.6, color=color, label=label, density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=7)
 
plt.tight_layout()
plt.show()

TyreLife, LapNumber and RaceProgress seems to carry most of the signal since sepration seems to be there on the basis of target

## CORRELATION HEATMAP

In [ ]:
corr_cols = num_cols + ['PitNextLap']
corr = train[corr_cols].corr()
target_corr = corr['PitNextLap'].drop('PitNextLap').sort_values()
print(target_corr.to_string())
 
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Correlations", color='white', fontsize=14)
 
# Full heatmap
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=axes[0], linewidths=0.5, annot_kws={'size': 7})
axes[0].set_title("Full Correlation Matrix")
 
# Target correlation bar
colors = [ACCENT if x > 0 else '#4e9af1' for x in target_corr.values]
axes[1].barh(target_corr.index, target_corr.values, color=colors)
axes[1].axvline(0, color='white', linewidth=0.8)
axes[1].set_title("Correlation with PitNextLap")
axes[1].set_xlabel("Pearson r")
 
plt.tight_layout()
plt.show()

**TyreLife** (0.27) and **LapNumber** (0.27) are the strongest positive predictors tied at the top. **Stint** (0.20) and **RaceProgress** (0.19) follow. Notably **Cumulative_Degradation** is negatively correlated (-0.17) which is counterintuitive it likely means the feature is engineered relative to a baseline and negative values signal degradation. **LapTime_Delta** is essentially flat at -0.005, meaning raw lap time delta alone is nearly useless as a linear predictor.

Also notice **RaceProgress** and **LapNumber** are correlated at 0.96, and **TyreLife** and **LapNumber** at 0.65 these three carry overlapping information. You don't need all three raw; they'll be somewhat redundant.

## CATEGORICAL ANALYSIS

In [ ]:
# Compound
compound_pit = train.groupby('Compound')['PitNextLap'].mean().sort_values(ascending=False)
print("\nPit rate by Compound:\n", compound_pit)
 
# Year
year_pit = train.groupby('Year')['PitNextLap'].mean().sort_values(ascending=False)
print("\nPit rate by Year:\n", year_pit)
 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Categorical Pit Rates", color='white', fontsize=14)
 
compound_pit.plot(kind='bar', ax=axes[0], color=ACCENT, edgecolor='white')
axes[0].set_title("Pit Rate by Compound")
axes[0].set_ylabel("PitNextLap mean")
axes[0].tick_params(axis='x', rotation=45)
 
year_pit.plot(kind='bar', ax=axes[1], color='#4e9af1', edgecolor='white')
axes[1].set_title("Pit Rate by Year")
axes[1].tick_params(axis='x', rotation=45)
 
# Compound volume
train['Compound'].value_counts().plot(kind='bar', ax=axes[2],color='#7c7c7c', edgecolor='white')
axes[2].set_title("Compound Frequency (train)")
axes[2].tick_params(axis='x', rotation=45)
 
plt.tight_layout()
plt.show()

This is the most surprising graph. HARD tyres have the highest pit rate (33%), not Softs (19%) which is completely counterintuitive. This likely means Hard tyres are used in longer stints and appear more in races where multiple stops happen, or it's a data generation artifact from the synthetic competition dataset. Do not assume domain logic holds here.

## PIT TIMING — WHEN DO STOPS HAPPEN?

In [ ]:
pits = train[train['PitNextLap'] == 1]
 
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("When Do Pit Stops Happen?", color='white', fontsize=14)
 
# By lap number
axes[0, 0].hist(pits['LapNumber'], bins=60, color=ACCENT, edgecolor='white', linewidth=0.3)
axes[0, 0].set_title("Pit Events by LapNumber")
axes[0, 0].set_xlabel("LapNumber")
 
# By race progress
axes[0, 1].hist(pits['RaceProgress'], bins=50, color='#f1a94e', edgecolor='white', linewidth=0.3)
axes[0, 1].set_title("Pit Events by RaceProgress")
axes[0, 1].set_xlabel("RaceProgress")
 
# By TyreLife
axes[1, 0].hist(pits['TyreLife'], bins=50, color='#4e9af1', edgecolor='white', linewidth=0.3)
axes[1, 0].set_title("Pit Events by TyreLife")
axes[1, 0].set_xlabel("TyreLife (laps on tyre)")
 
# TyreLife by Compound at pit time
for compound in pits['Compound'].unique():
    subset = pits[pits['Compound'] == compound]['TyreLife']
    axes[1, 1].hist(subset, bins=40, alpha=0.55, label=compound, density=True)
axes[1, 1].set_title("TyreLife at Pit — by Compound")
axes[1, 1].set_xlabel("TyreLife")
axes[1, 1].legend()
 
plt.tight_layout()
plt.show()

* **LapNumber:** Pit stops are spread fairly broadly from laps 5-55 with no sharp single peak this suggests mixed 1-stop and 2-stop strategies coexisting, which makes prediction harder since there's no clean "everyone pits at lap 25" pattern
* **RaceProgress:** Peaks around 0.35-0.6, tapering off after 0.7 most pits happen in the first two-thirds of the race, very few late stops
* **TyreLife:** Strong peak at 10-20 laps, rapidly declining after 30 most drivers pit before 30 laps on a tyre, so TyreLife >30 with no pit is a strong signal
* **TyreLife by Compound:** All compounds cluster tightly around 5-15 laps which is very early and the distributions heavily overlap across compounds. This means compound doesn't cleanly separate stint length in this dataset, unlike real F1 intuition

## DEGRADATION ANALYSIS

In [ ]:
# Average LapTime_Delta by TyreLife bin
bins = pd.cut(train['TyreLife'], bins=range(0, 55, 5))
deg_by_tyre = train.groupby(bins)['LapTime_Delta'].mean()
print("Avg LapTime_Delta by TyreLife bin:\n", deg_by_tyre)
 
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Degradation Patterns", color='white', fontsize=14)
 
deg_by_tyre.plot(kind='bar', ax=axes[0], color=ACCENT, edgecolor='white')
axes[0].set_title("Avg LapTime_Delta by TyreLife Bin")
axes[0].set_xlabel("TyreLife (laps)")
axes[0].set_ylabel("Mean LapTime_Delta (s)")
axes[0].tick_params(axis='x', rotation=45)
 
# Cumulative degradation vs pit
train.groupby('PitNextLap')['Cumulative_Degradation'].hist(
    bins=50, alpha=0.6, ax=axes[1],
    density=True, label=['No Pit', 'Pit'])
axes[1].set_title("Cumulative_Degradation by PitNextLap")
axes[1].set_xlabel("Cumulative_Degradation")
axes[1].legend(['No Pit', 'Pit'])
 
plt.tight_layout()
plt.show()

## TRAIN / TEST DRIFT SUMMARY

In [ ]:
shared_num = [c for c in num_cols if c in test.columns]
drift = pd.DataFrame({
    'train_mean': train[shared_num].mean(),
    'test_mean':  test[shared_num].mean(),
    'train_std':  train[shared_num].std(),
    'test_std':   test[shared_num].std(),
})
drift['mean_diff_%'] = ((drift['test_mean'] - drift['train_mean'])
                         / drift['train_mean'].abs().replace(0, np.nan) * 100).round(2)
print(drift.to_string())
 
fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Train vs Test Mean Drift (%)", color='white', fontsize=14)
colors = [ACCENT if abs(v) > 10 else '#4e9af1' for v in drift['mean_diff_%']]
ax.barh(drift.index, drift['mean_diff_%'], color=colors)
ax.axvline(0, color='white', linewidth=0.8)
ax.set_xlabel("% difference (test - train) / train")
plt.tight_layout()
plt.show()

All features are well under 10% drift maximum is Position_Change and LapTime_Delta at ~5.1%, everything else under 0.5%. No features need drift correction safe to use em all.

## RACE-LEVEL PIT RATE (top/bottom races)

In [ ]:
race_pit = (train.groupby('Race')['PitNextLap']
            .agg(['mean', 'count'])
            .sort_values('mean', ascending=False))
print("Top 10 races by pit rate:\n", race_pit.head(10))
print("\nBottom 10:\n", race_pit.tail(10))
 
fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle("Pit Rate by Race (sorted)", color='white', fontsize=14)
race_pit['mean'].sort_values().plot(kind='barh', ax=ax, color=ACCENT, edgecolor='none')
ax.set_xlabel("PitNextLap rate")
ax.set_ylabel("Race")
plt.tight_layout()
plt.show()

# F1 Pit Stop Prediction — EDA Final Conclusions

---

## 1. Dataset Overview
The training set contains **439,140 lap-level observations** across multiple F1 seasons (2022–2025), with 16 features covering tyre, timing, and positional data. The target `PitNextLap` is a binary label indicating whether a driver pitted on the following lap.

---

## 2. Class Imbalance
The dataset has an approximate **80:20 split** between non-pit and pit laps. While this is imbalanced, it is not severe enough to require aggressive resampling. Since the competition metric is **AUC**, the model only needs to rank pit laps above non-pit laps — raw probabilities don't need to be perfectly calibrated. A mild `scale_pos_weight=4` in LightGBM/XGBoost is sufficient as a starting point.

---

## 3. Feature Signal Strength
The strongest linear predictors of `PitNextLap` are:

| Feature | Pearson r | Interpretation |
|---|---|---|
| `TyreLife` | +0.27 | Older tyres → more likely to pit |
| `LapNumber` | +0.27 | Later in race → more likely to pit |
| `Stint` | +0.20 | Higher stint number → likely a later stop |
| `RaceProgress` | +0.19 | Further into race → more likely to pit |
| `Cumulative_Degradation` | -0.17 | Negative encoding — higher degradation signals pit |
| `LapTime_Delta` | ~0.00 | Essentially no linear signal on its own |

`LapNumber` and `RaceProgress` are correlated at **0.96** — they carry near-identical information. Similarly `TyreLife` and `LapNumber` overlap at 0.65. During feature engineering, interaction terms will be more valuable than using these raw features independently.

---

## 4. Counterintuitive Compound Behavior
The compound pit rate ordering **defies F1 domain logic**:

| Compound | Pit Rate |
|---|---|
| HARD | 32.8% |
| SOFT | 19.3% |
| INTERMEDIATE | 15.2% |
| MEDIUM | 10.1% |
| WET | 2.5% |

Hard tyres having the highest pit rate (vs Softs being lowest) suggests this is likely a **synthetic data artifact** — the dataset is generated and does not fully reflect real-world tyre strategy. The key takeaway: **do not apply F1 domain assumptions blindly**. Let the model learn the patterns directly from data.

---

## 5. The 2023 Anomaly — Critical Issue
Year 2023 has a pit rate of only **~0.96%** compared to ~28–30% for all other years. This is a severe anomaly — 2023 laps are almost entirely labelled as non-pit. Possible causes include incomplete data collection or a labelling issue for that season. **Action required before modeling:**
- Check how many 2023 rows exist in train vs test
- If 2023 appears only in train, consider dropping it entirely
- If it appears in test, it must be kept but treated carefully in validation

---

## 6. Pit Timing Patterns
- Pit stops are spread broadly from **laps 5–55** with no single dominant peak, indicating a mixture of 1-stop and 2-stop strategies coexisting in the data
- Most pits occur in the **first 70% of race progress** — very few late-race stops
- The majority of drivers pit with **10–20 laps on the tyre**, with stops becoming rare beyond 30 laps of tyre age
- Tyre life distributions at pit time **heavily overlap across compounds**, meaning compound alone is not a clean predictor of when a stop happens

---

## 7. Degradation Signal is Inverted
`LapTime_Delta` is **negative across all TyreLife bins**, with the most negative values on fresh tyres (0–5 laps: -7.4s). This means the feature encodes pace relative to some reference, not lap-over-lap degradation. Fresh tyres produce the fastest relative laps (most negative delta), and the signal flattens as tyres age. Traditional degradation intuition (older = slower = pit) does not directly apply to this feature as engineered. Rolling rate-of-change of `LapTime_Delta` will likely be more informative than the raw value.

---

## 8. No Train/Test Drift
All features show **under 6% mean drift** between train and test, with most under 1%. This is excellent — local cross-validation scores will be a reliable proxy for leaderboard performance. No distribution shift corrections or special encoding are needed.

---

## 9. Pre-Season Testing — Remove from Training
The dataset includes **Pre-Season Testing** as a "race" with 22,492 rows. Testing sessions have no strategic pit stop logic — teams pit freely for tyre programmes and car setup. This introduces noise that does not reflect race conditions. **Recommendation: drop all Pre-Season Testing rows before training.**

---

## 10. Race-Level Variability
Pit rates vary significantly by circuit:
- **Highest**: Chinese GP (38.9%), Monaco GP (35.7%), Spanish GP (32.0%) — safety car prone or high-degradation circuits
- **Lowest**: Mexico City GP (9.1%), Miami GP (10.4%), Italian GP (13.2%) — low-degradation, one-stop circuits

This variability suggests `Race` as a label-encoded or target-encoded feature could add meaningful signal, especially since circuit characteristics directly influence pit strategy.

---

## Summary — Key Actions Before Modeling

| Priority | Action |
|---|---|
| 🔴 High | Investigate and likely drop 2023 data |
| 🔴 High | Remove Pre-Season Testing rows |
| 🔴 High | Use year-based train/validation split — never random |
| 🟡 Medium | Encode `Compound` and `Race` (target encoding recommended) |
| 🟡 Medium | Fix `Cumulative_Degradation` plot (both classes rendered same color) |
| 🟢 Low | No drift correction needed — all features safe to use as-is |

---

*The dataset is clean, well-structured, and free of missing values. The main risks going into modeling are the 2023 anomaly, the presence of testing data, and over-reliance on domain assumptions that this synthetic dataset does not honour.*

# Baseline Modeling:

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

## Data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
 
print(f"Train shape : {train.shape}")
print(f"Test  shape : {test.shape}")

val_year   = 2025         # holding out most recent year as validation

## Encoding categorical

In [ ]:
cat_cols = ['Compound', 'Race', 'Driver']
 
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    # fitting on combined train+test to avoid unseen labels
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))
    le_dict[col] = le   #saving the le instance just in case required later
    print(f"Encoded {col}: {len(le.classes_)} unique values")

## Features

In [ ]:
drop_cols = ['id', 'PitNextLap', 'Year']  # Year used for split, not as feature
 
features = [c for c in train.columns if c not in drop_cols]
target   = 'PitNextLap'
 
print(f"\nFeatures ({len(features)}): {features}")

## Time-Aware train/val Split:
Thia validation strat was chosen because splitting blindly in this case would be dumb, signal could be lost since the data is lap based a lot of learnable information would never go under models eyes, thats why year based split is preffered by me. Maybe other strategies could have better results would have to test them later.

In [ ]:
val_mask   = train['Year'] == val_year
train_data = train[~val_mask]  # ~ means 'NOT'
val_data   = train[val_mask]
 
print(f"\nTrain rows : {len(train_data):,}  (years: {sorted(train_data['Year'].unique())})")
print(f"Val rows   : {len(val_data):,}    (year:  {val_year})")
print(f"Val positive rate: {val_data[target].mean():.3f}")
 
X_train = train_data[features]
y_train = train_data[target]
X_val   = val_data[features]
y_val   = val_data[target]
X_test  = test[features]

## LGBM BaseLine

In [ ]:
params = {
    'objective'        : 'binary',
    'metric'           : 'auc',
    'scale_pos_weight' : 4,       # accounts for ~80:20 imbalance
    'n_jobs'           : -1,
    'random_state'     : 42,
    'verbose'          : -1,
    'device'           : 'gpu'
}
 
model = lgb.LGBMClassifier(
    **params,
    n_estimators=1000,
)
 
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=100),
    ]
)

## Evaluation

In [ ]:
val_preds = model.predict_proba(X_val)[:, 1]
val_auc   = roc_auc_score(y_val, val_preds)

print(f"  Validation AUC : {val_auc:.5f}")
print(f"  Best iteration : {model.best_iteration_}")

## Feature Importance

In [ ]:
fi = pd.DataFrame({
    'feature'   : features,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)
 
print("\nTop 10 features by importance:")
print(fi.head(10).to_string(index=False))
 
plt.figure(figsize=(10, 6))
plt.barh(fi['feature'][:15][::-1], fi['importance'][:15][::-1], color='#e8002d')
plt.title('LightGBM Feature Importance (Top 15)', color='white')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Final Predictions and Submission

In [ ]:
test_preds = model.predict_proba(X_test)[:, 1]
 
submission = pd.DataFrame({
    'id'         : test['id'],
    target       : test_preds,
})
 
submission.to_csv('submission.csv', index=False)
print(f"Saved: submission.csv  ({len(submission):,} rows)")
print(f"\nPrediction stats:")
print(f"  Mean  : {test_preds.mean():.4f}")
print(f"  Std   : {test_preds.std():.4f}")
print(f"  Min   : {test_preds.min():.4f}")
print(f"  Max   : {test_preds.max():.4f}")

# Export backend artifacts

This section prepares the production baseline after the validation score above
has been accepted. It retrains the same LightGBM configuration on all available
training years using the validated best iteration.

The deployment format is intentionally a native LightGBM text model plus JSON
metadata. No pickle or joblib file is required. Run these cells only when you
are ready to generate the files for `backend/artifacts/`.


In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
import json
import platform
from pathlib import Path

# Works when the notebook starts from either the repository root or Experiments/.
project_root = Path.cwd().parent if Path.cwd().name == "Experiments" else Path.cwd()
artifact_dir = project_root / "backend" / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

model_path = artifact_dir / "baseline_lgbm.txt"
metadata_path = artifact_dir / "metadata.json"
smoke_test_path = artifact_dir / "smoke_test.json"

print(f"Artifacts will be written to: {artifact_dir.resolve()}")


## Retrain the production model

Early stopping selected the tree count using the 2025 validation set. The
production model below then learns from every available training row while
keeping that tree count fixed. This training still runs in the notebook—not on
the resource-constrained backend.


In [ ]:
best_iteration = int(model.best_iteration_ or model.n_estimators_)

production_model = lgb.LGBMClassifier(
    **params,
    n_estimators=best_iteration,
)
production_model.fit(train[features], train[target])

print(f"Production rows: {len(train):,}")
print(f"Trees: {best_iteration}")


## Save the model and prediction contract

`metadata.json` is part of the model contract. In particular, the backend must
apply `feature_order` exactly and use the stored categorical mappings. Unknown
Driver, Compound, or Race values should be rejected instead of assigned an
invented code.


In [ ]:
production_model.booster_.save_model(
    str(model_path),
    num_iteration=best_iteration,
)

category_mappings = {
    column: {
        str(label): int(index)
        for index, label in enumerate(le_dict[column].classes_.tolist())
    }
    for column in cat_cols
}

numeric_features = [column for column in features if column not in cat_cols]
numeric_ranges = {
    column: {
        "min": float(train[column].min()),
        "max": float(train[column].max()),
    }
    for column in numeric_features
}

model_bytes = model_path.read_bytes()
metadata = {
    "artifact_version": 1,
    "model_name": "baseline_lgbm",
    "model_format": "lightgbm_text",
    "model_file": model_path.name,
    "model_sha256": sha256(model_bytes).hexdigest(),
    "model_size_bytes": len(model_bytes),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_notebook": (
        "Experiments/lap-by-lap-s5e6-detailed-eda-baseline-lgbm.ipynb"
    ),
    "target": target,
    "feature_order": features,
    "model_feature_names": production_model.booster_.feature_name(),
    "numeric_features": numeric_features,
    "categorical_features": cat_cols,
    "categorical_mappings": category_mappings,
    "unknown_category_policy": "reject",
    "numeric_training_ranges": numeric_ranges,
    "decision_threshold": 0.5,
    "best_iteration": best_iteration,
    "training_rows": int(len(train)),
    "training_years": [int(year) for year in sorted(train["Year"].unique())],
    "validation": {
        "strategy": "hold_out_latest_year",
        "year": int(val_year),
        "roc_auc": float(val_auc),
    },
    "library_versions": {
        "python": platform.python_version(),
        "lightgbm": lgb.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}

metadata_path.write_text(
    json.dumps(metadata, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print(f"Saved {model_path.name}: {len(model_bytes) / 1024:.1f} KiB")
print(f"Saved {metadata_path.name}")


## Create and verify a smoke test

The smoke test records one raw API-style request and its expected probability.
It reloads the model from disk before predicting, so this cell catches broken
files, incorrect feature order, and incomplete categorical mappings.


In [ ]:
encoded_sample = X_val.iloc[[0]][features].copy()
raw_sample = {}

for column in features:
    value = encoded_sample.iloc[0][column]
    if column in cat_cols:
        raw_sample[column] = str(
            le_dict[column].inverse_transform([int(value)])[0]
        )
    elif isinstance(value, np.generic):
        raw_sample[column] = value.item()
    else:
        raw_sample[column] = value

reloaded_model = lgb.Booster(model_file=str(model_path))

# LightGBM may normalize feature names when saving (for example, replacing
# spaces with underscores). Compare with the in-memory booster names rather
# than the original DataFrame labels. The raw API contract still uses
# metadata["feature_order"].
expected_model_feature_names = production_model.booster_.feature_name()
assert reloaded_model.feature_name() == expected_model_feature_names
assert reloaded_model.num_feature() == len(features)

# Inference is positional, so use the exact feature order from the contract.
encoded_values = encoded_sample[features].to_numpy(dtype=np.float64)
expected_probability = float(
    reloaded_model.predict(
        encoded_values,
        num_iteration=best_iteration,
    )[0]
)
in_memory_probability = float(
    production_model.booster_.predict(
        encoded_values,
        num_iteration=best_iteration,
    )[0]
)

assert abs(expected_probability - in_memory_probability) < 1e-12
assert sha256(model_path.read_bytes()).hexdigest() == metadata["model_sha256"]

smoke_test = {
    "artifact_version": metadata["artifact_version"],
    "input": raw_sample,
    "expected": {
        "pit_next_lap_probability": expected_probability,
        "decision_threshold": metadata["decision_threshold"],
        "pit_next_lap": bool(
            expected_probability >= metadata["decision_threshold"]
        ),
    },
    "absolute_tolerance": 1e-10,
}
smoke_test_path.write_text(
    json.dumps(smoke_test, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print(f"Saved {smoke_test_path.name}")
print(f"Reloaded probability: {expected_probability:.10f}")
print("Artifact reload verification passed.")


## Files to copy

After the verification cell passes, place these files in
`backend/artifacts/` and commit them:

1. `baseline_lgbm.txt`
2. `metadata.json`
3. `smoke_test.json`

Do not copy the training CSV files, notebook state, or a pickled estimator into
the backend image.
